In [1]:
import pandas as pd

In [21]:
df = pd.read_csv("train_(2)_(1)_(1).csv")
print(df.head())

   ID             area_type   availability                  location  \
0   0  Super built-up  Area         19-Dec  Electronic City Phase II   
1   1            Plot  Area  Ready To Move          Chikka Tirupathi   
2   2        Built-up  Area  Ready To Move               Uttarahalli   
3   3  Super built-up  Area  Ready To Move        Lingadheeranahalli   
4   4  Super built-up  Area  Ready To Move                  Kothanur   

        size  society total_sqft  bath  balcony   price  
0      2 BHK  Coomee        1056   2.0      1.0   39.07  
1  4 Bedroom  Theanmp       2600   5.0      3.0  120.00  
2      3 BHK      NaN       1440   2.0      3.0   62.00  
3      3 BHK  Soiewre       1521   3.0      1.0   95.00  
4      2 BHK      NaN       1200   2.0      1.0   51.00  


In [23]:
c

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10656 entries, 0 to 10655
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            10656 non-null  int64  
 1   area_type     10656 non-null  object 
 2   availability  10656 non-null  object 
 3   location      10655 non-null  object 
 4   size          10642 non-null  object 
 5   society       6228 non-null   object 
 6   total_sqft    10656 non-null  object 
 7   bath          10591 non-null  float64
 8   balcony       10152 non-null  float64
 9   price         10656 non-null  float64
dtypes: float64(3), int64(1), object(6)
memory usage: 832.6+ KB
None


In [25]:
print(df.isnull().sum())

ID                 0
area_type          0
availability       0
location           1
size              14
society         4428
total_sqft         0
bath              65
balcony          504
price              0
dtype: int64


In [27]:
print(df.describe())

                 ID          bath       balcony         price
count  10656.000000  10591.000000  10152.000000  10656.000000
mean    5327.500000      2.690586      1.581757    112.214256
std     3076.266568      1.352403      0.821357    142.231967
min        0.000000      1.000000      0.000000      8.000000
25%     2663.750000      2.000000      1.000000     50.000000
50%     5327.500000      2.000000      2.000000     72.000000
75%     7991.250000      3.000000      2.000000    120.000000
max    10655.000000     40.000000      3.000000   2700.000000


In [29]:
print(df.shape)

(10656, 10)


In [31]:
#Handle missing values for bath numerical column
df['bath'] = df['bath'].fillna(df['bath'].median())

In [33]:
#Handle missing values for balcony numerical column
df['balcony'] = df['balcony'].fillna(df['balcony'].median())

In [35]:
#Handle missing values for Society categorical column
df['society'] = df['society'].fillna('Unknown')

In [37]:
#Missing value for location and size
df['location'] = df['location'].fillna(df['location'].mode()[0])

In [43]:
#missing values for size
df['size'] = df['size'].fillna(df['size'].mode()[0])

In [47]:
import re
import numpy as np

In [49]:

# Function to clean and convert total_sqft to numeric
def clean_total_sqft(sqft):
    try:
        # Handle numeric values or strings that are directly numeric
        sqft = str(sqft).strip()
        if sqft.replace('.', '').isdigit():
            return float(sqft)
        
        # Handle ranges (e.g., "2100 - 2850")
        if '-' in sqft:
            start, end = map(float, sqft.split('-'))
            return (start + end) / 2
        
        # Handle other units
        # Extract number and unit using regex
        match = re.match(r'(\d+\.?\d*)\s*([A-Za-z\s\.]+)?', sqft)
        if match:
            value = float(match.group(1))
            unit = match.group(2).lower().replace(' ', '') if match.group(2) else ''
            
            # Conversion factors
            if unit in ['sq.meter', 'sqmeter', 'sq.m', 'sqm']:
                return value * 10.7639  # Sq. Meter to Sq. Ft.
            elif unit in ['sq.yards', 'sqyard', 'sq.y', 'sqy']:
                return value * 9  # Sq. Yard to Sq. Ft.
            elif unit in ['acres', 'acre']:
                return value * 43560  # Acre to Sq. Ft.
            elif unit in ['perch']:
                return value * 272.25  # Perch to Sq. Ft.
            elif unit in ['cents', 'cent']:
                return value * 435.6  # Cent to Sq. Ft.
            elif unit in ['guntha']:
                return value * 1089  # Guntha to Sq. Ft.
            elif unit == '' or unit in ['sq.ft', 'sqft', 'squarefeet']:
                return value  # Already in Sq. Ft.
        
        # Return NaN for unparseable values
        return np.nan
    except:
        return np.nan

# Apply the cleaning function
df['total_sqft'] = df['total_sqft'].apply(clean_total_sqft)

In [51]:
# Check for missing values
print("Missing values in total_sqft before imputation:", df['total_sqft'].isnull().sum())

Missing values in total_sqft before imputation: 1


In [53]:
# Impute missing values with median
df['total_sqft'] = df['total_sqft'].fillna(df['total_sqft'].median())

In [55]:
print(df['total_sqft'].describe())

count    1.065600e+04
mean     1.971593e+03
std      1.913972e+04
min      1.000000e+00
25%      1.100000e+03
50%      1.275000e+03
75%      1.680000e+03
max      1.306800e+06
Name: total_sqft, dtype: float64


In [57]:
#Treating inconsistency in Size column and converting all to "x BHK"
def standardize_size(size):
    try:
        # Handle missing or invalid entries
        if pd.isna(size) or not isinstance(size, str) or size.strip() == '':
            return np.nan
        
        size = size.strip().lower()
        
        # Extract the number using regex
        match = re.match(r'(\d+)', size)
        if match:
            number = match.group(1)
            return f"{number} BHK"
        
        # Handle special cases
        if 'studio' in size:
            return "1 BHK"  # Assume Studio is 1 BHK
        if 'rk' in size:
            return f"{size[0]} BHK"  # e.g., "1 RK" → "1 BHK"
        
        # If no number is found, return NaN
        return np.nan
    except:
        return np.nan

In [59]:
# Apply the standardization
df['size'] = df['size'].apply(standardize_size)

In [61]:
# Check for missing values
print("Missing values in size before imputation:", df['size'].isnull().sum())

Missing values in size before imputation: 0


In [63]:
# Impute missing values with mode
df['size'] = df['size'].fillna(df['size'].mode()[0])

In [65]:
# Verify the conversion
print("\nUnique size values after standardization:")
print(df['size'].unique())
print("\nFirst few rows of size:")
print(df[['size']].head(10))
print("\nMissing values in size after imputation:", df['size'].isnull().sum())
print("\nValue counts of size:")
print(df['size'].value_counts())


Unique size values after standardization:
['2 BHK' '4 BHK' '3 BHK' '6 BHK' '1 BHK' '8 BHK' '7 BHK' '5 BHK' '11 BHK'
 '9 BHK' '27 BHK' '10 BHK' '19 BHK' '16 BHK' '43 BHK' '14 BHK' '12 BHK'
 '13 BHK']

First few rows of size:
    size
0  2 BHK
1  4 BHK
2  3 BHK
3  3 BHK
4  2 BHK
5  2 BHK
6  4 BHK
7  4 BHK
8  3 BHK
9  6 BHK

Missing values in size after imputation: 0

Value counts of size:
size
2 BHK     4441
3 BHK     3871
4 BHK     1137
1 BHK      536
5 BHK      282
6 BHK      175
7 BHK       77
8 BHK       68
9 BHK       48
10 BHK      10
11 BHK       4
27 BHK       1
19 BHK       1
16 BHK       1
43 BHK       1
14 BHK       1
12 BHK       1
13 BHK       1
Name: count, dtype: int64


In [67]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10656 entries, 0 to 10655
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            10656 non-null  int64  
 1   area_type     10656 non-null  object 
 2   availability  10656 non-null  object 
 3   location      10656 non-null  object 
 4   size          10656 non-null  object 
 5   society       10656 non-null  object 
 6   total_sqft    10656 non-null  float64
 7   bath          10656 non-null  float64
 8   balcony       10656 non-null  float64
 9   price         10656 non-null  float64
dtypes: float64(4), int64(1), object(5)
memory usage: 832.6+ KB
None


In [73]:
print(df['location'].value_counts())

location
Whitefield                434
Sarjapur  Road            303
Electronic City           234
Kanakpura Road            217
Thanisandra               178
                         ... 
Sahyadri Layout             1
Laxminarayana Layout        1
Kanakadasa Layout           1
Indira Nagar 3rd Stage      1
Panathur Road,              1
Name: count, Length: 1197, dtype: int64
